## 9. Code for position notation GUI

In [1]:
# Load position data from nwb (Not the processed)
# Plot them on the video

In [22]:
from spyglass.common.common_behav import RawPosition
from spyglass.common.common_behav import VideoFile
import pynwb
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

In [5]:
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename
from spyglass.shijiegu.decodeHelpers import runSessionNames
from spyglass.shijiegu.Analysis_SGU import ChangeofMindTheta

In [6]:
nwb_file_name = 'lewis20240109.nwb'
nwb_copy_file_name = get_nwb_copy_filename(nwb_file_name)

In [7]:
session_interval, position_interval = runSessionNames(nwb_copy_file_name)
session_name = session_interval[0]
pos_name = position_interval[0]

In [11]:
key = {"nwb_file_name": nwb_copy_file_name,"interval_list_name":pos_name}
raw_position = RawPosition.PosObject & key
spatial_series = raw_position.fetch_nwb()[0]["raw_position"]
spatial_df = raw_position.fetch1_dataframe()

[2025-11-22 20:12:14,521][WARNING]: Skipped checksum for file with hash: e630fc87-b027-ed25-8410-26bf159c7e37, and path: /stelmo/nwb/raw/lewis20240109_.nwb
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'core' version 2.6.0-alpha because version 2.7.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'ndx-franklab-novela' version 0.1.0 because version 0.2.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/pynwb/behavior.py:48: UserWarning: SpatialSeries 'series_0' has data shape (30647, 6) which is not compliant with NWB 2.5 and greater. The second dimension should have length <= 3 to 

In [ ]:
key = {'nwb_file_name': nwb_copy_file_name,'epoch':int(session_name[:2])}

raw_dir = '/stelmo/nwb/raw'

# video path
videoPath = VideoFile.get_abs_path(key)
video_info = (VideoFile & key).fetch1()

# parent underscore version nwb path
nwb_path = f"{raw_dir}/{video_info['nwb_file_name']}"

# load video timestamp from parent underscore version nwb
with pynwb.NWBHDF5IO(path=nwb_path, mode="r") as in_out:
    nwb_file = in_out.read()
    nwb_video = nwb_file.objects[video_info["video_file_object_id"]]
    video_filepath = VideoFile.get_abs_path(
        {"nwb_file_name": key["nwb_file_name"], "epoch": key["epoch"]}
    )
    video_dir = os.path.dirname(video_filepath) + "/"
    video_filename = video_filepath.split(video_dir)[-1]
    meters_per_pixel = nwb_video.device.meters_per_pixel
    timestamps = np.asarray(nwb_video.timestamps)

# load video
cap = cv2.VideoCapture(video_dir+video_filename)

/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'core' version 2.6.0-alpha because version 2.7.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'ndx-franklab-novela' version 0.1.0 because version 0.2.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/pynwb/behavior.py:48: UserWarning: SpatialSeries 'series_0' has data shape (30647, 6) which is not compliant with NWB 2.5 and greater. The second dimension should have length <= 3 to represent at most x, y, z.
  warnings.warn("SpatialSeries '%s' has data shape %s which is not compliant with NWB 2.5 and greater. "
/home/shijiegu/anaconda3

In [20]:
q = {"nwb_file_name": nwb_copy_file_name,
     "epoch":int(session_name[:2]),
     "proportion":str(0.1),
     "delta_t_minus":5,
     "delta_t_plus":5}

log_df = ChangeofMindTheta().fetch1_dataframe(q)
log_df_short = log_df[log_df.long_theta]
log_df_short

trialIDs = log_df_short.index
(t0,t1) = (log_df_short.loc[trialIDs[0]].timestamp_H, 
           log_df_short.loc[trialIDs[0]].timestamp_O)

In [21]:
frameToPlot = np.argwhere(np.logical_and(timestamps>=t0,timestamps<=t1)).ravel()
frame0 = frameToPlot[0]
frameLast = frameToPlot[-1]

fps = cap.get(cv2.CAP_PROP_FPS) # Gets the frames per second
frameSize = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))

fourcc = cv2.VideoWriter_fourcc(*'MP4V')
outputName = nwb_copy_file_name
out = cv2.VideoWriter(outputName+'_rawposition_layered.mp4', fourcc, fps, frameSize)

for fi in range(len(timestamps)):##len(frameToPlot)):
    t = timestamps[fi]

    ret, frame = cap.read()
    if fi >= frame0 and fi <= frameLast:
        # find rat position in cm to pixel
        pos_ind = np.argwhere(spatial_df.index >= t).ravel()[0]
        head_position_x = int(spatial_df.iloc[pos_ind].xloc) #in pixel
        head_position_y = int(spatial_df.iloc[pos_ind].yloc) #in pixel
    
        # convert rat to pixel
        plt.scatter(head_position_x, head_position_y,color = 'C0')
    
        if np.isnan(head_position_x) or np.isnan(head_position_y):
            continue
        cv2.circle(frame, (head_position_x, head_position_y), 3, (0, 0, 255), -1)
        #cv2.imshow("Scatter Plot on Frame", frame)
        out.write(frame)
    if fi > frameLast:
        break
        
cap.release()
out.release()

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'


NameError: name 'plt' is not defined